<a href="https://colab.research.google.com/github/dzidz1/Freeuni_ML_Facial_Expression_Recognition/blob/main/02_deeper_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup, data and training utilities

In [28]:
!pip install -q --upgrade kaggle wandb
import os, math, numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from google.colab import userdata
import wandb

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
wandb.login(key=userdata.get('WANDB_API_KEY'))
print("Device:", device)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


Device: cuda


In [29]:
!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge
!unzip -q -o challenges-in-representation-learning-facial-expression-recognition-challenge.zip

challenges-in-representation-learning-facial-expression-recognition-challenge.zip: Skipping, found more recently modified local copy (use --force to force download)


In [30]:
df = pd.read_csv('icml_face_data.csv')
df.columns = df.columns.str.strip()
df['Usage'] = df['Usage'].str.strip()

def pixels_to_array(s):
  return np.array([np.array(p.split(), dtype=np.uint8) for p in s]).reshape(-1, 48, 48)

class FERDataset(Dataset):
  def __init__(self, images, labels, transform=None):
    self.images, self.labels, self.transform = images, labels, transform
  def __len__(self):
    return len(self.images)
  def __getitem__(self, idx):
    img = torch.from_numpy(self.images[idx]).float().unsqueeze(0) / 255.0
    if self.transform:
      img = self.transform(img)
    return img, int(self.labels[idx])

splits = {}
for name, usage in [('train', 'Training'), ('val', 'PublicTest'), ('test', 'PrivateTest')]:
  sub = df[df['Usage'] == usage]
  splits[name] = FERDataset(pixels_to_array(sub['pixels']), sub['emotion'].values)

BATCH = 64
train_loader = DataLoader(splits['train'], batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(splits['val'],   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(splits['test'],  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print("Train/Val/Test:", len(splits['train']), len(splits['val']), len(splits['test']))

Train/Val/Test: 28709 3589 3589


In [31]:
def train_one_epoch(model, loader, criterion, optimizer, device):
  model.train(); rl, c, t = 0.0, 0, 0
  for x, y in loader:
    x, y = x.to(device), y.to(device)
    optimizer.zero_grad()
    out = model(x); loss = criterion(out, y)
    loss.backward(); optimizer.step()
    rl += loss.item()*x.size(0); c += (out.argmax(1)==y).sum().item(); t += y.size(0)
  return rl/t, c/t

@torch.no_grad()
def evaluate(model, loader, criterion, device):
  model.eval(); rl, c, t = 0.0, 0, 0
  for x, y in loader:
    x, y = x.to(device), y.to(device)
    out = model(x); loss = criterion(out, y)
    rl += loss.item()*x.size(0); c += (out.argmax(1)==y).sum().item(); t += y.size(0)
  return rl/t, c/t

import copy

def run_experiment(model, run_name, group, config, train_loader, val_loader, device, patience=None):
  wandb.init(project="fer2013-emotion-recognition", name=run_name, group=group, config=config)
  model = model.to(device)
  criterion = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], weight_decay=config.get("weight_decay", 0))

  best_val_loss, best_val_acc, best_state, best_epoch, wait = float('inf'), 0.0, None, 0, 0
  for epoch in range(1, config["epochs"]+1):
    tl, ta = train_one_epoch(model, train_loader, criterion, optimizer, device)
    vl, va = evaluate(model, val_loader, criterion, device)
    wandb.log({"epoch": epoch, "train_loss": tl, "train_acc": ta, "val_loss": vl, "val_acc": va})
    print(f"Epoch {epoch:2d}/{config['epochs']} | train {tl:.4f}/{ta:.4f} | val {vl:.4f}/{va:.4f}")
    best_val_acc = max(best_val_acc, va)
    if vl < best_val_loss:
      best_val_loss, best_state, best_epoch, wait = vl, copy.deepcopy(model.state_dict()), epoch, 0
    else:
      wait += 1
      if patience and wait >= patience:
        print(f"Early stopping at epoch {epoch} (best val_loss was epoch {best_epoch})")
        break

  if best_state is not None:
    model.load_state_dict(best_state)
  wandb.run.summary.update({"best_val_acc": best_val_acc, "best_val_loss": best_val_loss, "best_epoch": best_epoch})
  wandb.finish()
  return model

## Architecture - Deeper CNN with BatchNorm

In [32]:
class DeeperCNN(nn.Module):
    def __init__(self, num_classes=7, dropout=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),   nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),  nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 6 * 6, 256), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))


In [ ]:
config = {"architecture": "DeeperCNN", "lr": 1e-3, "batch_size": 64,
          "optimizer": "Adam", "weight_decay": 0, "dropout": 0.0, "epochs": 30}
model = DeeperCNN(dropout=config["dropout"])
run_experiment(model, run_name="DeeperCNN-overfit", group="DeeperCNN",
               config=config, train_loader=train_loader, val_loader=val_loader, device=device)

Epoch  1/30 | train 1.6130/0.3702 | val 1.3662/0.4870
Epoch  2/30 | train 1.2981/0.5030 | val 1.2587/0.5127
Epoch  3/30 | train 1.1888/0.5462 | val 1.1995/0.5450
Epoch  4/30 | train 1.0980/0.5871 | val 1.1652/0.5525
Epoch  5/30 | train 1.0166/0.6158 | val 1.1443/0.5637
Epoch  6/30 | train 0.9399/0.6447 | val 1.1683/0.5690
Epoch  7/30 | train 0.8682/0.6728 | val 1.2784/0.5497
Epoch  8/30 | train 0.7841/0.7081 | val 1.2907/0.5439
Epoch  9/30 | train 0.6973/0.7440 | val 1.2436/0.5907
Epoch 10/30 | train 0.6099/0.7777 | val 1.4191/0.5684
Epoch 11/30 | train 0.5361/0.8045 | val 1.4350/0.5709
Epoch 12/30 | train 0.4451/0.8396 | val 1.4761/0.5807
Epoch 13/30 | train 0.3716/0.8680 | val 1.5798/0.5548
Epoch 14/30 | train 0.3171/0.8896 | val 1.6996/0.5731
Epoch 15/30 | train 0.2571/0.9101 | val 1.9177/0.5709
Epoch 16/30 | train 0.2152/0.9276 | val 1.8985/0.5818
Epoch 17/30 | train 0.1760/0.9407 | val 2.0329/0.5762
Epoch 18/30 | train 0.1722/0.9402 | val 2.2151/0.5656
Epoch 19/30 | train 0.1524/0

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_acc,▁▃▃▄▄▄▄▅▅▆▆▆▇▇▇▇██████████████
train_loss,█▇▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▃▅▅▆▇▅▅█▆▇▇▆▇▇▇▇▆▆▇▇▇▅▇▇▇▇▆▇▅
val_loss,▂▁▁▁▁▁▁▂▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▆▇▇▇▇██
epoch,30
train_acc,0.97523
train_loss,0.07835
val_acc,0.54862
val_loss,3.11632


DeeperCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4608, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropou

In [ ]:
config = {"architecture": "DeeperCNN", "lr": 1e-3, "batch_size": 64, "optimizer": "Adam",
          "weight_decay": 1e-4, "dropout": 0.5, "epochs": 40}
model = DeeperCNN(dropout=config["dropout"])
run_experiment(model, run_name="DeeperCNN-regularized", group="DeeperCNN",
               config=config, train_loader=train_loader, val_loader=val_loader, device=device, patience=7)

Epoch  1/40 | train 1.7414/0.3000 | val 1.5212/0.4291
Epoch  2/40 | train 1.5212/0.3987 | val 1.4073/0.4709
Epoch  3/40 | train 1.4558/0.4257 | val 1.4654/0.4244
Epoch  4/40 | train 1.4044/0.4446 | val 1.3554/0.4909
Epoch  5/40 | train 1.3619/0.4617 | val 1.2992/0.5127
Epoch  6/40 | train 1.3377/0.4737 | val 1.2502/0.5277
Epoch  7/40 | train 1.2961/0.4889 | val 1.2501/0.5068
Epoch  8/40 | train 1.2686/0.4943 | val 1.2334/0.5305
Epoch  9/40 | train 1.2364/0.5086 | val 1.2473/0.5277
Epoch 10/40 | train 1.2141/0.5166 | val 1.2361/0.5258
Epoch 11/40 | train 1.1777/0.5297 | val 1.2638/0.5210
Epoch 12/40 | train 1.1486/0.5440 | val 1.1760/0.5581
Epoch 13/40 | train 1.1195/0.5546 | val 1.1898/0.5405
Epoch 14/40 | train 1.0850/0.5692 | val 1.1847/0.5550
Epoch 15/40 | train 1.0545/0.5815 | val 1.2178/0.5453
Epoch 16/40 | train 1.0201/0.5918 | val 1.2211/0.5545
Epoch 17/40 | train 0.9898/0.6058 | val 1.2376/0.5659
Epoch 18/40 | train 0.9570/0.6189 | val 1.2308/0.5578
Epoch 19/40 | train 0.9308/0

epoch,▁▁▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇██
train_acc,▁▃▄▄▄▅▅▅▅▆▆▆▆▇▇▇███
train_loss,█▆▆▅▅▅▄▄▄▃▃▃▃▂▂▂▂▁▁
val_acc,▁▃▁▄▅▆▅▆▆▆▆█▇▇▇▇██▇
val_loss,█▆▇▅▃▃▃▂▂▂▃▁▁▁▂▂▂▂▂
best_epoch,12
best_val_acc,0.5659
best_val_loss,1.17601
epoch,19
train_acc,0.62684
train_loss,0.93082


DeeperCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4608, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropou

In [33]:
config = {"architecture": "DeeperCNN", "lr": 1e-3, "batch_size": 64, "optimizer": "Adam",
          "weight_decay": 1e-2, "dropout": 0.8, "epochs": 30}
model = DeeperCNN(dropout=config["dropout"])
run_experiment(model, run_name="DeeperCNN-underfit", group="DeeperCNN",
               config=config, train_loader=train_loader, val_loader=val_loader, device=device)

Epoch  1/30 | train 1.8536/0.2345 | val 1.7339/0.3062
Epoch  2/30 | train 1.7388/0.2869 | val 1.6306/0.3288
Epoch  3/30 | train 1.6521/0.3257 | val 1.5353/0.4093
Epoch  4/30 | train 1.5391/0.3890 | val 1.4262/0.4430
Epoch  5/30 | train 1.4688/0.4223 | val 1.4096/0.4486
Epoch  6/30 | train 1.4216/0.4453 | val 1.4176/0.4536
Epoch  7/30 | train 1.3842/0.4652 | val 1.3399/0.4884
Epoch  8/30 | train 1.3572/0.4770 | val 1.3587/0.4893
Epoch  9/30 | train 1.3339/0.4862 | val 1.3322/0.4884
Epoch 10/30 | train 1.3188/0.4925 | val 1.3348/0.4795
Epoch 11/30 | train 1.3095/0.4998 | val 1.2921/0.5107
Epoch 12/30 | train 1.2982/0.5030 | val 1.2627/0.5180
Epoch 13/30 | train 1.2811/0.5111 | val 1.2954/0.5021
Epoch 14/30 | train 1.2745/0.5162 | val 1.2421/0.5269
Epoch 15/30 | train 1.2658/0.5185 | val 1.2750/0.5185
Epoch 16/30 | train 1.2629/0.5211 | val 1.2404/0.5316
Epoch 17/30 | train 1.2598/0.5238 | val 1.2945/0.5029
Epoch 18/30 | train 1.2543/0.5207 | val 1.2673/0.5138
Epoch 19/30 | train 1.2466/0

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_acc,▁▂▃▅▅▆▆▇▇▇▇▇▇▇▇███████████████
train_loss,█▇▆▅▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▂▄▅▅▅▆▆▆▆▇▇▇▇▇█▇▇▇▇█▆███▇▇▇▇▇
val_loss,█▇▅▄▄▄▃▃▃▃▂▂▂▁▂▁▂▂▂▂▁▃▁▂▁▂▁▂▂▂
best_epoch,21
best_val_acc,0.54862
best_val_loss,1.20538
epoch,30
train_acc,0.53628
train_loss,1.21784


DeeperCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4608, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropou

In [34]:
from torchvision import transforms

train_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(12),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
])

aug_train_ds = FERDataset(splits['train'].images, splits['train'].labels, transform=train_aug)
aug_train_loader = DataLoader(aug_train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)

In [35]:
config = {"architecture": "DeeperCNN", "lr": 1e-3, "batch_size": 64, "optimizer": "Adam",
          "weight_decay": 1e-4, "dropout": 0.3, "epochs": 50, "augmentation": True}
model = DeeperCNN(dropout=config["dropout"])
run_experiment(model, run_name="DeeperCNN-augmented", group="DeeperCNN",
               config=config, train_loader=aug_train_loader, val_loader=val_loader, device=device, patience=10)

Epoch  1/50 | train 1.7488/0.2970 | val 1.5321/0.4113
Epoch  2/50 | train 1.5627/0.3823 | val 1.3898/0.4720
Epoch  3/50 | train 1.4748/0.4237 | val 1.3327/0.4890
Epoch  4/50 | train 1.4367/0.4424 | val 1.3135/0.4912
Epoch  5/50 | train 1.3902/0.4646 | val 1.2621/0.5035
Epoch  6/50 | train 1.3659/0.4703 | val 1.3225/0.4940
Epoch  7/50 | train 1.3381/0.4872 | val 1.2253/0.5288
Epoch  8/50 | train 1.3165/0.4969 | val 1.2674/0.5216
Epoch  9/50 | train 1.2997/0.5034 | val 1.2072/0.5433
Epoch 10/50 | train 1.2849/0.5108 | val 1.1764/0.5609
Epoch 11/50 | train 1.2702/0.5156 | val 1.1725/0.5564
Epoch 12/50 | train 1.2641/0.5156 | val 1.1462/0.5595
Epoch 13/50 | train 1.2425/0.5268 | val 1.1601/0.5603
Epoch 14/50 | train 1.2389/0.5298 | val 1.1411/0.5779
Epoch 15/50 | train 1.2200/0.5389 | val 1.2344/0.5266
Epoch 16/50 | train 1.2239/0.5359 | val 1.2056/0.5489
Epoch 17/50 | train 1.2072/0.5439 | val 1.1182/0.5843
Epoch 18/50 | train 1.2036/0.5466 | val 1.0836/0.5846
Epoch 19/50 | train 1.1935/0

epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
train_acc,▁▃▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇███████████
train_loss,█▆▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁
val_acc,▁▃▄▄▄▅▅▅▆▆▆▅▅▇▇▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▇█▇▇██▇██
val_loss,█▆▅▅▄▄▅▄▃▃▃▃▄▄▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▁▁▂▂▁▁▁▁▂▁
best_epoch,49
best_val_acc,0.63249
best_val_loss,0.99433
epoch,50
train_acc,0.63182
train_loss,0.98575


DeeperCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4608, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropou